In [11]:
# =========================================================
# REPO STATIC ANALYZER FOR KAGGLE/JUPYTER COMPATIBILITY
# =========================================================
# This scans:
# 1. hardcoded paths
# 2. os.chdir usage
# 3. nvidia-smi / GPU assumptions
# 4. API/server dependencies
# 5. argparse/main execution assumptions
# 6. suspicious notebook-breaking patterns
#
# SAFE:
# - READ ONLY
# - DOES NOT MODIFY FILES
# =========================================================

import os
import re
from collections import defaultdict

REPO_DIR = os.getcwd()   # run from repo root

print("=" * 80)
print("SCANNING REPOSITORY")
print("ROOT:", REPO_DIR)
print("=" * 80)

# ---------------------------------------------------------
# PATTERNS TO SEARCH
# ---------------------------------------------------------

PATTERNS = {
    "PATHS": [
        r'os\.chdir',
        r'open\(',
        r'os\.path\.join',
        r'os\.path\.abspath',
        r'prompt/',
        r'checkpoints/',
        r'saved_models/',
        r'logs/',
        r'video/',
    ],

    "GPU/NVIDIA": [
        r'nvidia-smi',
        r'torch\.cuda',
        r'cuda',
        r'DEVICE',
        r'gpu',
    ],

    "ONLINE_LLM/API": [
        r'requests\.',
        r'FastAPI',
        r'uvicorn',
        r'zhipuai',
        r'httpx',
        r'api_key',
        r'requests\.post',
    ],

    "CLI/NOTEBOOK_ISSUES": [
        r'argparse',
        r'sys\.argv',
        r'if __name__ == .__main__.:',
        r'input\(',
    ],

    "FILE_IO": [
        r'pickle',
        r'json\.load',
        r'json\.dump',
        r'torch\.save',
        r'torch\.load',
    ],

    "MULTIPROCESSING": [
        r'multiprocessing',
        r'Subproc',
        r'threading',
        r'Process',
    ],
}

# ---------------------------------------------------------
# STORAGE
# ---------------------------------------------------------

results = defaultdict(list)

# ---------------------------------------------------------
# WALK THROUGH FILES
# ---------------------------------------------------------

for root, dirs, files in os.walk(REPO_DIR):

    # skip hidden/cache folders
    dirs[:] = [
        d for d in dirs
        if d not in ['.git', '__pycache__', '.ipynb_checkpoints']
    ]

    for file in files:

        if not file.endswith(".py"):
            continue

        filepath = os.path.join(root, file)

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                lines = f.readlines()
        except Exception as e:
            print(f"Could not read {filepath}: {e}")
            continue

        for line_num, line in enumerate(lines, start=1):

            for category, patterns in PATTERNS.items():

                for pattern in patterns:

                    if re.search(pattern, line):

                        relpath = os.path.relpath(filepath, REPO_DIR)

                        results[category].append({
                            "file": relpath,
                            "line": line_num,
                            "text": line.strip()
                        })

# ---------------------------------------------------------
# PRINT RESULTS
# ---------------------------------------------------------

for category, matches in results.items():

    print("\n" + "=" * 80)
    print(f"{category}")
    print("=" * 80)

    if not matches:
        print("No matches found.")
        continue

    for m in matches:
        print(f"\n[{m['file']} : line {m['line']}]")
        print(m['text'])

# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

for category, matches in results.items():
    print(f"{category:<25} -> {len(matches)} matches")

print("\nDONE.")

SCANNING REPOSITORY
ROOT: c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main

PATHS

[Game.py : line 26]
task_info_json = os.path.join(prefix, "prompt/task_info.json")

[Game.py : line 26]
task_info_json = os.path.join(prefix, "prompt/task_info.json")

[Game.py : line 42]
model_dir = os.path.join(args.logdir, args.policy, args.task, args.loaddir, args.loadmodel)

[Game.py : line 83]
with open(task_info_json, 'r') as f:

[Game.py : line 255]
dir_path = os.path.join(self.logger.dir, dir_name)

[Game.py : line 296]
video_path = os.path.join(dir_path, video_name)

[algos\base.py : line 39]
torch.save(self.model, os.path.join(self.save_path, name + filetype))

[utils\log.py : line 24]
output_dir = os.path.join(args.logdir, args.policy, args.task, args.savedir)

[utils\log.py : line 26]
output_dir = os.path.join(args.logdir, args.policy, args.task, args.loaddir, args.savedir)

[utils\log.py : line 31]
info_path = os.path.join(output_dir, "config.json")

[utils\log.py : line 33]
with ope

In [1]:
!pip install gymnasium minigrid torch numpy opencv-python torch_ac


In [3]:
# ============================================================
# FULL RELOAD + SANITY TESTS  (paste this as ONE cell)
# ============================================================
import sys, os, importlib

# --- make sure repo root is on path ---
REPO = r"C:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

# --- force-reload every relevant module so stale cache can't bite ---
for mod_name in list(sys.modules.keys()):
    if any(x in mod_name for x in ["Game","algos","skill","env","mediator","teacher"]):
        del sys.modules[mod_name]

# ── 1. imports ──────────────────────────────────────────────
import Game as _G; print("Game file:", _G.__file__)
import torch
print("torch :", torch.__version__)
print("cuda  :", torch.cuda.is_available())

# ── 2. build args ───────────────────────────────────────────
from types import SimpleNamespace
args = SimpleNamespace(
    seed=0, task="SimpleDoorKey", frame_stack=1,
    offline_planner=True, soft_planner=False,
    logdir="log", policy="ppo",
    loaddir=None, loadmodel="acmodel", savedir="debug_run",
    device="cpu", batch_size=32, recurrent=False,
    gamma=0.99, lam=0.95,
    n_itr=1, traj_per_itr=1,
    num_eval=1, eval_interval=1, save_interval=1,
)

# ── 3. init ──────────────────────────────────────────────────
from Game import Game
game = Game(args)
print("\n[1] INIT OK")

# ── 4. teacher eval ──────────────────────────────────────────
out = game.evaluate(teacher_policy=True, record_frames=False, deterministic=False)
print("[2] TEACHER EVAL OK  →", out)

# ── 5. student eval ──────────────────────────────────────────
out = game.evaluate(teacher_policy=False, record_frames=False, deterministic=False)
print("[3] STUDENT EVAL OK  →", out)

# ── 6. standalone collect ────────────────────────────────────
game.collect()
print("[4] COLLECT OK  buffer len =", len(game.buffer))

# ── 7. train ─────────────────────────────────────────────────
game.train()
print("[5] TRAIN OK")

print("\n✅  ALL 5 TESTS PASSED")

c:\Users\HP\anaconda3\envs\gpt_inf\Lib\site-packages\gymnasium\envs\registration.py:637: UserWarning: WARN: Overriding environment MiniGrid-SimpleDoorKey-Min5-Max10-View3 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\HP\anaconda3\envs\gpt_inf\Lib\site-packages\gymnasium\envs\registration.py:637: UserWarning: WARN: Overriding environment MiniGrid-LavaDoorKey-Min5-Max10-View3 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\HP\anaconda3\envs\gpt_inf\Lib\site-packages\gymnasium\envs\registration.py:637: UserWarning: WARN: Overriding environment MiniGrid-ColoredDoorKey-Min5-Max10-View3 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\HP\anaconda3\envs\gpt_inf\Lib\site-packages\gymnasium\envs\registration.py:637: UserWarning: WARN: Overriding environment MiniGrid-TwoDoor-Min20-Max20 already in registry.
  logger.warn(f"

Game file: C:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main\Game.py
torch : 2.11.0+cpu
cuda  : False
[INFO]: resetting the task: SimpleDoorKey
Logging to log\ppo\SimpleDoorKey\debug_run
use MLP......

[1] INIT OK
[2] TEACHER EVAL OK  → (0.94, 10, 1)
[3] STUDENT EVAL OK  → (0.0, 150, 0)
[4] COLLECT OK  buffer len = 150
********** Iteration 0 ************
time elapsed: 0.00 s
0.95 s to collect    150 timesteps | 157.35sample/s.
2.20 s to optimizer| loss 14.436, entropy  1.381, kickstarting  1.416.
-------------------------------------------------
|                 Timesteps |             150 |
|            Return (train) |             0.0 |
|    Episode Length (train) |           150.0 |
|      Success Rate (train) |             0.0 |
-------------------------------------------------
[5] TRAIN OK

✅  ALL 5 TESTS PASSED
